Name: Shaun Clarke

Course: CSC6314 Deep Learning

Instructor: Margaret Mulhall

Assignment: Project #2: The Optimization & Regularization Challenge

In this project, you will implement and compare a traditional Machine Learning system (Logistic Regression) against a Deep Learning system (Multi-Layer Perceptron) using PyTorch. Submit a single .py file.


Project Objective The goal of this assignment is to understand how different training configurations and architectural enhancements impact a model's ability to learn. You will build a deep MLP using the Keras Sequential API and train it across two separate experimental runs. By comparing the outputs, you will observe how a combination of an adaptive optimizer, normalization, and learning rate scheduling influences convergence speed and final accuracy compared to a baseline Stochastic Gradient Descent (SGD) model.

In [11]:
from typing import Tuple, List, Dict
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.models import Sequential
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
import sklearn
from sklearn import metrics
# The dataset i will be working with
from sklearn.datasets import fetch_california_housing
# Split the data into training and testing
from sklearn.model_selection import train_test_split
# For standardizing ML features so they are all on the same scale mean=0 std=1
from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import accuracy_score


In [12]:
##### Requirement 1  Data Engineering: The Setup#####
# Reused most of this from project 1.
# This function loads and preps the data
def load_and_prep_data() -> Tuple[np.ndarray, np.ndarray, np.ndarray,np.ndarray,np.ndarray]:
    """
    """

    "Step1 : Loading the data"
    # Loading the housing dataset
    data: sklearn.utils._bunch.Bunch = fetch_california_housing()
    # Defining the features/inputs before preprocessing, hence the raw
    x_raw: np.ndarray = data.data
    # Defining the target variables that we are trying to predict. not yet converted to binary
    y_raw = data.target

    "Step 2: creating binary labels"
    # Creating the binary labels by converting all values in Y_raw above the 70th percentile ar represented as false
    # and teh rest labeld as true. then convert the boolean to binary 1,0.
    threshold: np.ndarray = np.percentile(y_raw, 70)
    y_binary: np.ndarray = (y_raw > threshold).astype(
        int)  # 1 = high value, 0 = not high value

    "Step 3: Train/Test split"
    # Splitting x_raw and y_binary into train test splits, using stratify to make sure the class distribution is preserved for bothe the training and test sets.
    # Uisng 0.95 as the test_size for this week because we want training to be 5% and testing to be 95% on the split.
    X_train, X_test, Y_train, Y_test = train_test_split(
        x_raw, y_binary, test_size=0.95, random_state=42, stratify=y_binary)

    "Step 4: Instantiate StandardScaler"
    # initializin StandardScaler
    scaler: StandardScaler = StandardScaler()
    # fitting and transforming the training data with standarscaler
    X_train_scaled: np.ndarray = scaler.fit_transform(X_train)
    # We will only perform a transform on the test data, to avoid data leakage(exposing test data to the model)
    X_test_scaled: np.ndarray = scaler.transform(X_test)

    # Getting the number of columns for the input dimensions from the tuple returned by .shape (num_rows, num_cols) num_columns).
    # each column\feature is an input and each row is a sample.
    # The imput_dim tells tf how many features to expect for each sample.
    # Basically means, input_dim tells torch the number of input it's getting per row, which determines the size of the input layer.
    # Then the first layer creates the weights based on the size of the input layer.
    input_dim: int = X_train_scaled.shape[1]

    return X_train_scaled, X_test_scaled, Y_train, Y_test, input_dim



In [13]:
X_train, X_test, Y_train, Y_test, input_dim = load_and_prep_data()

In [14]:
# This function spits out a 5 layer MLP with adn can turn batch normalization on and off
def build_model(batch_norm: bool, optimizer: Adam, input_dim) -> tf.keras.models:
  """
  """

  # Initializing the model
  model = Sequential()

  # List of layer widths(size of each neuron layer) to loop over
  layer_sizes: List = [256, 128, 64, 32, 16]

  # Iterating over the layer_sizes and generating the inpu tlayer and the 5 hidden layers.
  for i, units in enumerate(layer_sizes):

    # if this is the frist layer then add input_shape
    if i == 0:
      # model.add(Dense(units=units, kernel_initializer="he_normal", activation="relu", kernel_regularizer=l2(0.001), input_shape=(input_dim,)))
      model.add(Input(shape=(input_dim,)))

    model.add(Dense(units=units, kernel_initializer="he_normal", activation="relu", kernel_regularizer=l2(0.001)))

    # if batch_norm is True add batch normalization layer
    if batch_norm:
      model.add(BatchNormalization())

    # Adding dropout layer
    model.add(Dropout(0.3))

  # Adding the output layer
  model.add(Dense(units=1, kernel_initializer="he_normal",activation="sigmoid",  kernel_regularizer=l2(0.001)))

  # Compiling the model
  model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])

  return model




In [15]:
# Building strategy 1 model
# strategy_1_model = build_model(batch_norm=False, optimizer=SGD(learning_rate=0.001), input_dim=input_dim)

In [16]:
# strategy_1_model.summary()

In [17]:
# Building strategy 2 model
# strategy_2_model = build_model(batch_norm=True, optimizer=Adam(learning_rate=0.001),  input_dim=input_dim)

In [18]:
# This method trains the model
def train_model(
    model: tf.keras.models,
    X_train: np.ndarray,
    Y_train: np.ndarray,
    callbacks: List
) -> tf.keras.callbacks.History:

  """
  """

  return model.fit(
      X_train,
      Y_train,
      batch_size=32,
      validation_split=0.2,
      epochs=100,
      verbose=1,
      callbacks=callbacks
  )


In [19]:
# This fucntion evaluates both models and prints a comparison table using pandas.
# More straightforward than print formatiing, in my opinion that is.
def evaluate_print_results(
    models_dict: Dict,
    histories_dict: Dict,
    X_test: np.ndarray,
    Y_test: np.ndarray
) -> None:

  """
  """

  # Getting Model 1 and Model 2 from dictionary
  strategy_1_model = models_dict["strategy_1_model"]
  strategy_2_model = models_dict["strategy_2_model"]

  # Getting  Model 1 and Model 2 history objects from dictionaries
  model_1_history = histories_dict["strategy_1_model"]
  model_2_history = histories_dict["strategy_2_model"]

  # Evcaluating model 1 and model 2. To get loss and accuracy and supressing the output with verbose=0
  evaluate_model_1 = strategy_1_model.evaluate(X_test, Y_test, verbose=0)
  evaluate_model_2 =  strategy_2_model.evaluate(X_test, Y_test, verbose=0)

  # Getting the test accuracy from evaluate results. index 1 is the accruracy
  test_accuracy_1 = evaluate_model_1[1]
  test_accuracy_2 = evaluate_model_2[1]

  # Getting the final training accuracy from history. The last epoch, index -1
  train_accuracy_1 = model_1_history["accuracy"][-1]
  train_accuracy_2 = model_2_history["accuracy"][-1]



  # Creating comparison data_frame. Each key is a row label
  data = {
        "Metric": [
            "Epochs to Converge",
            "Final Train Loss",
            "Final Test Accuracy",
            "Generalization Gap"
        ],
        "1. Standard (SGD)": [
            len(model_1_history['loss']),
            round(model_1_history['loss'][-1], 4),
            f"{test_accuracy_1 * 100:.2f}%",
            round(train_accuracy_1 - test_accuracy_1, 4)
        ],
        "2. Advanced  (Adam + BN + Sched)": [
            len(model_2_history['loss']),
            round(model_2_history['loss'][-1], 4),
            f"{test_accuracy_2 * 100:.2f}%",
            round(train_accuracy_2 - test_accuracy_2, 4)
        ]
    }

  # Generatingt he dataframe for the comparison table
  df = pd.DataFrame(data)

  # Using metric as the index so rows are labeled, and not numbered
  df = df.set_index("Metric")

  print("\n===== Model Comparison Results =====")
  # Converting to string to print full table. Before convertng to string it was being cut off
  print(df.to_string())

In [20]:
def main() -> None:

  # Loading data
  X_train, X_test, Y_train, Y_test, data_input_dim = load_and_prep_data()

  ##### MODEL 1 #####
  # preparing model 1 hyperparameters
  sgd_optimizer = SGD(learning_rate=0.001)
  early_stopping = EarlyStopping(monitor='val_loss', patience=5)

  # Building strategy 1 model
  strategy_1_model = build_model(batch_norm=False, optimizer=sgd_optimizer, input_dim=data_input_dim)
  print(strategy_1_model.summary())
  # training strategy 1 model
  trained_model_1 = train_model(strategy_1_model, X_train, Y_train, callbacks=[early_stopping])

  ##### MODEL 2 #####
  # preparing model 2 hyperparameters
  adam_optimizer = Adam(learning_rate=0.001)
  early_stopping = EarlyStopping(monitor='val_loss', patience=5)
  reduce_on_plateau = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

  # Building strategy 2 model
  strategy_2_model = build_model(batch_norm=True, optimizer=adam_optimizer, input_dim=data_input_dim)
  print(strategy_2_model.summary())
  # training strategy 2 model
  trained_model_2 = train_model(strategy_2_model, X_train, Y_train, callbacks=[early_stopping, reduce_on_plateau])

  # Creating dictionaries for the models and the model histories
  histories_dict = {
      "strategy_1_model": trained_model_1.history,
      "strategy_2_model": trained_model_2.history
  }

  models_dict = {
      "strategy_1_model": strategy_1_model,
      "strategy_2_model": strategy_2_model
  }

  # Evaluating the model
  evaluate_print_results(models_dict, histories_dict, X_test, Y_test)


if __name__ == "__main__":
  main()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 256)            │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,081 (180.00 KB)

 Trainable params: 46,081 (180.00 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 256)            │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,065 (187.75 KB)

 Trainable params: 47,073 (183.88 KB)

 Non-trainable params: 992 (3.88 KB)

None
Epoch 1/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.5455 - loss: 1.7983 - val_accuracy: 0.7440 - val_loss: 1.5242 - learning_rate: 0.0010
Epoch 2/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6582 - loss: 1.6079 - val_accuracy: 0.7778 - val_loss: 1.4721 - learning_rate: 0.0010
Epoch 3/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7067 - loss: 1.5307 - val_accuracy: 0.7923 - val_loss: 1.4415 - learning_rate: 0.0010
Epoch 4/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7321 - loss: 1.4780 - val_accuracy: 0.8019 - val_loss: 1.4093 - learning_rate: 0.0010
Epoch 5/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7321 - loss: 1.4923 - val_accuracy: 0.7923 - val_loss: 1.3977 - learning_rate: 0.0010
Epoch 6/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7806 - loss: 1.4267 - val_accuracy: 0.7971 - val_loss: 1.3696 - learning_rate: 0.0010
Epoch 7/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8012 - loss: 1.3798